# Incrementality Testing Case Study

## Did the campaign create value—or capture demand that already existed?

This case study analyzes a randomized customer holdout experiment for a promotional campaign. It estimates causal conversion lift, incremental revenue, incremental return on ad spend (iROAS), and heterogeneous effects across customer segments.

### Business decision

A marketing team spent **$18,000** targeting eligible customers. Management must decide whether to scale, redesign, or stop the campaign.

The analysis answers:

1. Did treatment increase conversion?
2. How many conversions and dollars were genuinely incremental?
3. Did incremental value exceed campaign cost?
4. Which customer segments responded most strongly?

> Attribution measures credited outcomes. Incrementality measures outcomes that would not have occurred without treatment.

## 1. Setup

The notebook creates deterministic synthetic experiment data, so it runs without an external dataset. Replace the simulation with a warehouse extract that has one row per randomized unit.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

RANDOM_STATE = 42
CAMPAIGN_COST = 18_000
PRIMARY_METRIC = "converted"
SEGMENTS = [
    "VIP Customers",
    "Affluent but Unengaged",
    "Core Customers",
    "Promising Spenders",
    "Budget Conscious",
]

## 2. Experiment design

Eligible customers are randomized 50/50 before exposure:

- **Treatment:** receives the campaign.
- **Control:** receives no campaign.
- **Primary outcome:** conversion within the measurement window.
- **Secondary outcome:** realized revenue.
- **Estimand:** intention-to-treat effect—the impact of assignment, whether or not every assigned customer saw the message.

Randomization makes the control group's outcome a credible estimate of what would have happened to treated customers without the campaign.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
n_customers = 20_000

segment = rng.choice(
    SEGMENTS,
    size=n_customers,
    p=[0.15, 0.18, 0.42, 0.13, 0.12],
)
treatment = rng.binomial(1, 0.5, n_customers)

baseline_by_segment = {
    "VIP Customers": 0.34,
    "Affluent but Unengaged": 0.11,
    "Core Customers": 0.23,
    "Promising Spenders": 0.27,
    "Budget Conscious": 0.10,
}
lift_by_segment = {
    "VIP Customers": 0.010,
    "Affluent but Unengaged": 0.060,
    "Core Customers": 0.035,
    "Promising Spenders": 0.045,
    "Budget Conscious": 0.008,
}

baseline_probability = np.array([baseline_by_segment[s] for s in segment])
true_lift = np.array([lift_by_segment[s] for s in segment])
conversion_probability = baseline_probability + treatment * true_lift
converted = rng.binomial(1, conversion_probability)

order_value = rng.gamma(shape=4.0, scale=24.0, size=n_customers)
revenue = converted * order_value

experiment = pd.DataFrame({
    "customer_id": np.arange(1, n_customers + 1),
    "segment": segment,
    "treatment": treatment,
    "converted": converted,
    "revenue": revenue,
})
display(experiment.head())

## 3. Data-quality and randomization checks

In [ ]:
assert experiment["customer_id"].is_unique
assert experiment.isna().sum().sum() == 0
assert set(experiment["treatment"].unique()) == {0, 1}
assert set(experiment["converted"].unique()) == {0, 1}

allocation = experiment["treatment"].value_counts(normalize=True).sort_index()
segment_balance = pd.crosstab(
    experiment["segment"], experiment["treatment"], normalize="columns"
)

print("Treatment allocation:")
display(allocation.rename(index={0: "Control", 1: "Treatment"}).to_frame("share"))
print("Segment composition by group:")
display(segment_balance.style.format("{:.1%}"))

Randomization checks are diagnostic rather than hypothesis tests to optimize. Large imbalances can indicate assignment or logging failures; small chance differences are expected.

## 4. Overall experiment results

In [ ]:
group_summary = experiment.groupby("treatment").agg(
    customers=("customer_id", "count"),
    conversions=("converted", "sum"),
    conversion_rate=("converted", "mean"),
    total_revenue=("revenue", "sum"),
    revenue_per_customer=("revenue", "mean"),
).rename(index={0: "Control", 1: "Treatment"})

display(group_summary.style.format({
    "conversion_rate": "{:.2%}",
    "total_revenue": "${:,.0f}",
    "revenue_per_customer": "${:,.2f}",
}))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
colors = ["#64748b", "#0f766e"]

axes[0].bar(group_summary.index, group_summary["conversion_rate"], color=colors)
axes[0].set(title="Conversion rate", ylabel="Rate")
axes[0].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")

axes[1].bar(group_summary.index, group_summary["revenue_per_customer"], color=colors)
axes[1].set(title="Revenue per assigned customer", ylabel="Revenue")
axes[1].yaxis.set_major_formatter(lambda value, _: f"${value:.0f}")

fig.suptitle("Treatment outperforms control on both outcomes", fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Estimate causal lift and uncertainty

For two independent proportions, the estimated standard error is:

$$SE(\hat p_T-\hat p_C)=\sqrt{\frac{\hat p_T(1-\hat p_T)}{n_T}+\frac{\hat p_C(1-\hat p_C)}{n_C}}$$

The confidence interval communicates plausible effect sizes; the p-value evaluates compatibility with a zero-lift null hypothesis.

In [ ]:
p_t = group_summary.loc["Treatment", "conversion_rate"]
p_c = group_summary.loc["Control", "conversion_rate"]
n_t = group_summary.loc["Treatment", "customers"]
n_c = group_summary.loc["Control", "customers"]

absolute_lift = p_t - p_c
relative_lift = absolute_lift / p_c
standard_error = np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
z_score = absolute_lift / standard_error
p_value = 2 * (1 - norm.cdf(abs(z_score)))
ci_low = absolute_lift - 1.96 * standard_error
ci_high = absolute_lift + 1.96 * standard_error

effect_summary = pd.DataFrame({
    "metric": ["Absolute lift", "Relative lift", "95% CI lower", "95% CI upper", "p-value"],
    "value": [absolute_lift, relative_lift, ci_low, ci_high, p_value],
})
display(effect_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.errorbar(
    x=absolute_lift,
    y=0,
    xerr=[[absolute_lift - ci_low], [ci_high - absolute_lift]],
    fmt="o",
    color="#0f766e",
    ecolor="#0f766e",
    capsize=7,
    markersize=9,
)
ax.axvline(0, color="#dc2626", linestyle="--", label="No effect")
ax.set(xlabel="Absolute conversion lift", yticks=[], title="Estimated lift with 95% confidence interval")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Translate lift into business value

Incremental conversions estimate how many treated conversions would not have occurred under control. Incremental revenue values those conversions at the control group's average order value, avoiding credit for treatment-side order-value differences that were not separately tested.

$$\text{iROAS}=\frac{\text{incremental revenue}}{\text{campaign cost}}$$

In [ ]:
control_aov = experiment.loc[
    (experiment["treatment"] == 0) & (experiment["converted"] == 1), "revenue"
].mean()

incremental_conversions = absolute_lift * n_t
incremental_revenue = incremental_conversions * control_aov
incremental_profit_before_fixed_cost = incremental_revenue - CAMPAIGN_COST
iroas = incremental_revenue / CAMPAIGN_COST

business_results = pd.DataFrame({
    "metric": [
        "Incremental conversions",
        "Incremental revenue",
        "Campaign cost",
        "Net incremental value",
        "iROAS",
    ],
    "value": [
        incremental_conversions,
        incremental_revenue,
        CAMPAIGN_COST,
        incremental_profit_before_fixed_cost,
        iroas,
    ],
})
display(business_results)

## 7. Segment-level treatment effects

Overall lift can hide meaningful heterogeneity. Because treatment was randomized, lift can be estimated within pre-existing customer segments. These subgroup results should be treated as exploratory unless the analysis and multiplicity correction were pre-specified.

In [ ]:
segment_rates = (
    experiment.groupby(["segment", "treatment"])["converted"]
    .agg(["count", "mean"])
    .reset_index()
)
segment_lift = (
    segment_rates.pivot(index="segment", columns="treatment", values="mean")
    .rename(columns={0: "control_rate", 1: "treatment_rate"})
)
segment_lift["absolute_lift"] = segment_lift["treatment_rate"] - segment_lift["control_rate"]
segment_lift["relative_lift"] = segment_lift["absolute_lift"] / segment_lift["control_rate"]
segment_lift = segment_lift.sort_values("absolute_lift", ascending=False)

display(segment_lift.style.format("{:.2%}"))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
bar_colors = ["#0f766e" if value >= absolute_lift else "#64748b" for value in segment_lift["absolute_lift"]]
ax.barh(segment_lift.index, segment_lift["absolute_lift"], color=bar_colors)
ax.invert_yaxis()
ax.axvline(0, color="#111827", linewidth=1)
ax.axvline(absolute_lift, color="#d97706", linestyle="--", label="Overall lift")
ax.set(xlabel="Absolute conversion lift", title="Campaign impact differs by customer segment")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.1%}")
ax.legend()
plt.tight_layout()
plt.show()

## 8. Recommendation

Use the computed results—not attributed conversions—to make the decision:

- **Scale selectively** if the confidence interval excludes zero and net incremental value is positive.
- Prioritize segments with large, credible lift, especially **Affluent but Unengaged** and **Core Customers** in this simulation.
- Avoid subsidizing naturally high converters when their incremental response is small.
- Re-test at higher spend levels before extrapolating iROAS; marginal returns commonly decline.
- Monitor margin, unsubscribe rate, long-run retention, and cross-channel spillovers as guardrails.

This connects to the [Customer Segmentation](https://github.com/JCZY999/Customer_Segmentation), [A/B Testing](https://github.com/JCZY999/A_B_Testing), and [Marketing Mix Modeling](https://github.com/JCZY999/Marketing-Mix-Modeling) projects.

## 9. Export reusable outputs

In [ ]:
OUTPUT_DIR = Path("outputs/notebook_case_study")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

experiment.to_csv(OUTPUT_DIR / "experiment_data.csv", index=False)
group_summary.to_csv(OUTPUT_DIR / "group_summary.csv")
effect_summary.to_csv(OUTPUT_DIR / "effect_summary.csv", index=False)
business_results.to_csv(OUTPUT_DIR / "business_results.csv", index=False)
segment_lift.to_csv(OUTPUT_DIR / "segment_lift.csv")

for path in sorted(OUTPUT_DIR.glob("*.csv")):
    print(f"Created: {path}")

## 10. Limitations and next steps

- The dataset is simulated and intentionally contains a positive treatment effect.
- Revenue uncertainty deserves its own inference or bootstrap; the primary test here is conversion.
- Segment estimates are noisier than the overall result and may require multiplicity adjustment.
- Noncompliance, missing outcomes, sample-ratio mismatch, and interference must be investigated in production.
- iROAS is valid only for the tested audience, creative, spend level, channel, season, and measurement window.
- Long-run incrementality can differ from short-run lift because of pull-forward and customer learning.

### Conclusion

A randomized holdout turns marketing measurement into a causal decision. The campaign should be judged by incremental conversions, uncertainty, incremental profit, and heterogeneous response—not by attributed revenue alone.